# Internal Variability in Climate Projections

**Internal variability** is one of three sources of uncertainty in climate
projections, alongside **model uncertainty** (different GCMs give different
answers) and **scenario uncertainty** (different emissions pathways give
different answers). It arises from the inherently chaotic behavior of the
climate system itself — interactions among the atmosphere, ocean, land
surface, and sea ice produce natural fluctuations that are not predictable
beyond weather timescales.

Two runs of the *same* model with the *same* forcing, differing only in
their initial conditions (different ensemble "members"), will diverge over
time purely due to this internal variability. It is distinct from
*external* natural forcing (volcanic eruptions, solar fluctuations) because
it originates *inside* the climate system, and is most apparent at
**regional scales** and **short time horizons** (seasonal to decadal). At
multi-decadal scales, model and scenario uncertainty progressively dominate.

Internal variability matters most for:
- **Extremes** (e.g. 99th-percentile precipitation) rather than means
- **Regional / local** scales rather than global
- **Near-term** projections (next few decades) rather than end-of-century
- Variables with naturally **high year-to-year variance**, like
  precipitation, more so than temperature

**This notebook walks through both halves of the problem:**
1. **Visualizing** internal variability — how large is it, and how does it
   compare to inter-model spread? (Steps 1–2)
2. **Handling** internal variability correctly in a statistical analysis —
   deseasonalizing, pooling, and significance testing (Steps 3–4)

> **For a deeper scientific treatment**, see the
> [IPCC AR5 uncertainty guidance](https://www.ipcc.ch/site/assets/uploads/2017/08/AR5_Uncertainty_Guidance_Note.pdf),
> [Hawkins & Sutton (2009)](https://journals.ametsoc.org/view/journals/bams/90/8/2009bams2607_1.xml),
> and [climatewest.ca: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/).

> **Note:** This notebook is part of a series. For inter-model spread see
> `model_uncertainty.ipynb`; for a full decomposition across all three
> uncertainty sources see `uncertainty_decomposition.ipynb`.

**Runtime:** Approximately **13–18 minutes** with default settings.


## Step 0: Setup

All imports are organized here in one place: standard library, scientific
stack, plotting (both cartopy and holoviews/panel), then climakitae-specific
modules. The two statistical helper functions this notebook depends on —
`_deseasonalize_monthly` and `get_ks_pval_df` — are defined later, directly
in this notebook (Step 3), rather than imported from an external module.
This keeps the notebook self-contained and easy to share or adapt.

This notebook uses `climakitae`'s `new_core` `ClimateData` interface for
the Step 1 WRF retrieval, which replaces the older
`DataParameters`/`CmipOpt`/`get_warm_level` pattern with a single fluent
query builder and a built-in `warming_level` process. The older
`DataParameters` interface is still used later (Step 1b, Step 3) where the
raw multi-member CMIP6 catalog or full-resolution WRF retrieval is not yet
exposed through `ClimateData`.


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import warnings

# ── Scientific stack ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats
from scipy.stats import gaussian_kde
import os
import gc
import pandas as pd
import numpy as np
import xarray as xr
from scipy import stats

# ── Plotting: cartopy (static, publication-style maps) ───────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ── Plotting: holoviews + panel (interactive, in-notebook exploration) ───
import holoviews as hv
import panel as pn
from bokeh.models import HoverTool
hv.extension("bokeh")
pn.extension()

# ── climakitae ─────────────────────────────────────────────────────────────
import climakitae as ck
from climakitae.util.colormap import read_ae_colormap
from climakitae.new_core.user_interface import ClimateData
from climakitae.core.data_interface import DataParameters
from climakitae.explore.uncertainty import GWL_1981_2010_FILE, read_csv_file
import climakitae as ck

# -- Custom uncertainty code ------------------------------------------------
from uncertainty import (
    new_core_get_ensemble_data,
    get_warm_level
)

warnings.filterwarnings("ignore")
# %config InlineBackend.figure_format = "svg"

## Step 1: Retrieve data

We use **monthly precipitation** because precipitation has among the
highest internal variability of any climate variable — making it the
clearest case for illustrating why internal variability matters and how to
handle it.

### Step 1a: Downscaled WRF precipitation (Cal-Adapt models)

This gives us 8 dynamically downscaled models over California at 9 km
resolution, historical + SSP 3-7.0. Each model here has exactly **one**
ensemble member — so this dataset captures **inter-model spread**, not
within-model internal variability. We'll come back to this distinction
directly in Step 2.


In [ ]:
cd = ClimateData()
cd.catalog("cadcat")
cd.variable("prec")
cd.table_id("mon")
cd.activity_id("WRF")
cd.institution_id("UCLA")
cd.experiment_id(["historical", "ssp370"])
cd.grid_label("d02")
cd.processes({"filter_unadjusted_models": "no"})  # keep all 8 Cal-Adapt models

wrf_ds = cd.get()
wrf_ds = wrf_ds.clip(0.1)  # remove near-zero noise

In [ ]:
warm_level = 3.0 # set your target

# ── Historical slice (unchanged logic) ───────────────────────────────────────
hist_wrf = wrf_ds.sel(time=slice('1981', '2010'))

# ── Future: warming-level slice via new_core (replaces get_warm_level loop) ──
cd_ssp = ClimateData()
cd_ssp.catalog('cadcat')
cd_ssp.variable('prec')
cd_ssp.table_id('mon')
cd_ssp.activity_id('WRF')
cd_ssp.institution_id('UCLA')
cd_ssp.experiment_id(['historical', 'ssp370'])
cd_ssp.grid_label('d02')
cd_ssp.processes({'filter_unadjusted_models': 'no'})
cd_ssp.processes({'warming_levels': warm_level})
ssp_wrf = cd_ssp.get()
ssp_wrf

In [ ]:
# Align to a common set of simulations and load into memory
common_sims = [s for s in ssp_wrf.sim.values if s in hist_wrf.sim.values]
ssp_wrf  = ssp_wrf.sel(sim=common_sims)
hist_wrf = hist_wrf.sel(sim=common_sims)

ssp_wrf  = ck.load(ssp_wrf.unify_chunks(),  progress_bar=True)
hist_wrf = ck.load(hist_wrf.unify_chunks(), progress_bar=True)
wrf_ds   = ck.load(wrf_ds.unify_chunks(),   progress_bar=True)

cads_hist_percentile  = ck.load(hist_wrf.chunk({"time": -1}).quantile([0.99], dim="time").squeeze(), progress_bar=True)
cads_ssp_percentile   = ck.load(ssp_wrf.chunk({"time": -1}).quantile([0.99], dim="time").squeeze(), progress_bar=True)
cads_delta_percentile = ck.load(cads_ssp_percentile - cads_hist_percentile, progress_bar=True)

### Step 1b: CMIP6 ensemble members for the same models

For each of the eight Cal-Adapt models, raw CMIP6 provides multiple
**ensemble members** — runs of the same model with slightly different
initial conditions. The spread across these members captures genuine
internal variability, since the model structure and forcing are identical
within each model — only the starting conditions differ. This dataset is
what lets us visualize internal variability directly in Step 2.

We use the older `new_core_get_ensemble_data` helper for this step since the
raw multi-member CMIP6 catalog is not yet exposed through `ClimateData` —
internally it still relies on `intake-esm` and a global-warming-level lookup
table, with `selections` (a `DataParameters` object) used only for spatial
clipping to California.


In [ ]:
selections = DataParameters()
selections.area_average = "No"
selections.area_subset = "states"
selections.cached_area = ["CA"]
selections.downscaling_method = "Dynamical"
selections.scenario_historical = ["Historical Climate"]
selections.scenario_ssp = ["SSP 3-7.0"]
selections.append_historical = True
selections.variable = "Precipitation (total)"
selections.time_slice = (1981, 2100)
selections.resolution = "9 km"
selections.timescale = "monthly"

cmip_names = [
    "EC-Earth3", "EC-Earth3-Veg", "CESM2", "CNRM-ESM2-1", "FGOALS-g3",
    "MIROC6", "TaiESM1", "MPI-ESM1-2-HR",
]

In [ ]:
hist_cae_ds, warm_cae_ds = new_core_get_ensemble_data(
    variable="pr",
    selections=selections,  # used only for spatial clipping to California
    cmip_names=cmip_names,
    warm_level=warm_level,
)

In [ ]:
hist_cae_ds = ck.load(hist_cae_ds, progress_bar=True)
warm_cae_ds = ck.load(warm_cae_ds, progress_bar=True)

In [ ]:
# 99th percentile per ensemble member, historical and future
hist_cae_p99  = ck.load(hist_cae_ds.quantile(0.99, dim="time").squeeze(), progress_bar=True)
warm_cae_p99  = ck.load(warm_cae_ds.quantile(0.99, dim="time").squeeze(), progress_bar=True)
delta_cae_p99 = ck.load(warm_cae_p99 - hist_cae_p99, progress_bar=True)

## Step 2: Visualize internal variability

### Step 2a: Within-model ensemble spread — spatial maps

Each row below shows all ensemble members for one model. Because the model
structure and forcing are identical across members, differences between
maps **within a row** are internal variability alone, while differences
**between rows** (different models) mix internal variability with model
uncertainty.

Two versions of this figure are provided: a static holoviews/bokeh grid
(interactive within the notebook) and a Plotly version that exports to a
standalone HTML file and a static PNG for use outside the notebook.


In [ ]:
def make_precip_map(data, title, vmin, vmax, diverging=False,
                     width=220, height=220):
    """Single precipitation map using holoviews QuadMesh."""
    cmap = read_ae_colormap(
        cmap="ae_diverging_r" if diverging else "ae_blue", cmap_hex=True
    )
    hover = HoverTool(tooltips=[
        ("Lon (\u00b0E)", "@x"), ("Lat (\u00b0N)", "@y"), ("Precip (mm/mo)", "@z")
    ])
    return hv.QuadMesh((data["lon"], data["lat"], data)).opts(
        tools=[hover], colorbar=True, cmap=cmap,
        symmetric=diverging, clim=(vmin, vmax),
        xaxis=None, yaxis=None,
        clabel="mm / month",
        title=title, width=width, height=height,
    )

In [ ]:
# Plot change in 99th-percentile precipitation per ensemble member, grouped by model
# Rename spatial dims to lon/lat for the map helper
delta_plot_ds = delta_cae_p99.rename({"x": "lon", "y": "lat"})
all_panels = None
for sim in np.unique(delta_plot_ds.simulation.values):
    sim_data = delta_plot_ds.where(delta_plot_ds.simulation == sim, drop=True)
    for mid in range(len(sim_data.member_id.values)):
        panel = make_precip_map(
            data=sim_data.pr.drop("simulation").isel(member_id=mid),
            title=f"{sim}  member {mid + 1}",
            vmin=-300, vmax=300,
            diverging=True,
        )
        all_panels = panel if all_panels is None else all_panels + panel

(
    all_panels.cols(6)
    .opts(
        hv.opts.Layout(
            merge_tools=True,
            toolbar="below",
            title=f"Change in 99th-percentile monthly precipitation at {warm_level}\u00b0C warming \u2014 per ensemble member"
        )
    )
)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import cartopy.feature as cfeature

MAX_MODELS = 7
MAX_MEMBERS = 4

sim_values = delta_plot_ds.simulation.values

# ── Order models by member count, descending (most members on the left) ────
member_counts = pd.Series(sim_values).value_counts()
unique_sims = member_counts.index[:MAX_MODELS].tolist()

panel_specs = []
for col_idx, sim in enumerate(unique_sims):
    member_indices = np.where(sim_values == sim)[0][:MAX_MEMBERS]
    for row_idx, mem_idx in enumerate(member_indices):
        panel_specs.append((sim, col_idx, row_idx, delta_plot_ds.isel(member_id=mem_idx)))

n_cols = len(unique_sims)
n_rows = MAX_MEMBERS

# Column titles at row 1; row labels ("Member 1", etc.) handled separately via annotations
subplot_titles = [""] * (n_rows * n_cols)
for col_idx, sim in enumerate(unique_sims):
    subplot_titles[col_idx] = sim

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.008,
    vertical_spacing=0.02,
)

vmin, vmax = -300, 300
cmap_hex = read_ae_colormap(cmap="ae_diverging_r", cmap_hex=True)
colorscale = [[i / (len(cmap_hex) - 1), c] for i, c in enumerate(cmap_hex)]

coastline_feature = cfeature.NaturalEarthFeature("physical", "coastline", "10m")
states_feature = cfeature.NaturalEarthFeature("cultural", "admin_1_states_provinces_lines", "10m")

def _geom_to_xy(geometries, extent):
    xs, ys = [], []
    lon_min, lon_max, lat_min, lat_max = extent
    for geom in geometries:
        for line in getattr(geom, "geoms", [geom]):
            coords = list(line.coords)
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
            if max(lons) < lon_min or min(lons) > lon_max or max(lats) < lat_min or min(lats) > lat_max:
                continue
            xs.extend(lons + [None])
            ys.extend(lats + [None])
    return xs, ys

extent = [
    float(delta_plot_ds.lon.min()), float(delta_plot_ds.lon.max()),
    float(delta_plot_ds.lat.min()), float(delta_plot_ds.lat.max()),
]
lon_span = extent[1] - extent[0]
lat_span = extent[3] - extent[2]
panel_aspect = lat_span / lon_span

coast_x, coast_y = _geom_to_xy(coastline_feature.geometries(), extent)
state_x, state_y = _geom_to_xy(states_feature.geometries(), extent)

for sim, col_idx, row_idx, panel_data in panel_specs:
    row = row_idx + 1
    col = col_idx + 1
    is_first = (row == 1 and col == 1)

    data = panel_data["pr"]

    fig.add_trace(
        go.Heatmap(
            x=data["lon"].values,
            y=data["lat"].values,
            z=data.values,
            colorscale=colorscale,
            zmin=vmin, zmax=vmax,
            showscale=is_first,
            colorbar=dict(
                title=dict(text="mm / month", side="top", font=dict(size=10)),
                orientation="h",
                len=0.65,          # increased from 0.5
                thickness=16,      # slightly thicker too, scales nicely with len
                x=0.5,
                xanchor="center",
                y=-0.05,
                yanchor="top",
                tickfont=dict(size=9),
            ) if is_first else None,
            hovertemplate=f"{sim}<br>Lon: %{{x:.2f}}<br>Lat: %{{y:.2f}}<br>Precip: %{{z:.1f}} mm/mo<extra></extra>",
        ),
        row=row, col=col,
    )

    fig.add_trace(
        go.Scatter(x=coast_x, y=coast_y, mode="lines",
                   line=dict(color="black", width=0.6),
                   showlegend=False, hoverinfo="skip"),
        row=row, col=col,
    )
    fig.add_trace(
        go.Scatter(x=state_x, y=state_y, mode="lines",
                   line=dict(color="#777777", width=0.4),
                   showlegend=False, hoverinfo="skip"),
        row=row, col=col,
    )

    fig.update_xaxes(showticklabels=False, range=[extent[0], extent[1]], row=row, col=col)
    fig.update_yaxes(showticklabels=False, range=[extent[2], extent[3]], row=row, col=col)

panel_width = 130
panel_height = panel_width * panel_aspect

fig.update_layout(
    title=dict(
        text=f"Change in 99th-percentile monthly precipitation at {warm_level}\u00b0C warming \u2014 per ensemble member",
        font=dict(size=14),
    ),
    height=int(panel_height * n_rows * 1.15) + 100,
    width=int(panel_width * n_cols * 1.2),
    plot_bgcolor="white",
    margin=dict(t=60, l=70, r=10, b=80),  # increased l (left margin) for row labels
)
fig.update_annotations(font_size=10)

# ── Row labels: "Member 1", "Member 2", ... on the far left of each row ─────
for row_idx in range(n_rows):
    fig.add_annotation(
        text=f"Member {row_idx + 1}",
        xref="paper", yref="paper",
        x=-0.,
        y=1 - (row_idx + 0.5) / n_rows,
        showarrow=False,
        font=dict(size=11, color="#333333"),
        xanchor="right",
        yanchor="middle",
    )

fig.show()
fig.write_html("../uncertainty/figures/html/members_variability.html")
fig.write_image("../uncertainty/figures/static/members_variability.png", scale=2)

**Reading the maps:** Focus on variation *within* a single row first.
Some models show one member with a strong wetting signal in the Sierra
Nevada and another member with near-zero or even negative change in the
same region — despite identical model physics and identical forcing. This
within-row spread is internal variability. In several cases, the within-row
spread is comparable in magnitude to the between-row (inter-model) spread —
which is the central finding this notebook sets out to quantify in Step 2b.


### Step 2b: Internal variability vs. inter-model spread — range chart

The spatial maps above show the *pattern*; this chart makes the comparison
between internal variability and inter-model spread directly *quantitative*.
For each model:
- The **blue bar** spans the full range of 99th-percentile precipitation
  changes across all CMIP6 ensemble members for that model — this range is
  internal variability
- The **horizontal tick** marks the ensemble median
- The **gold dot** marks the single downscaled WRF member available for
  that model — one draw from the blue distribution

All values are area-averaged over a representative sub-region of
Northern/Central California (defined just below) and expressed as % change
from the 1981–2010 baseline.


In [ ]:
# Define sub-region -- modify to your area of interest
lat0, lat1 = 36.0, 39.0
lon0, lon1 = -123.0, -119.5

In [ ]:
wrf_da = wrf_ds["prec"].clip(0.1)

# ── Spatial subset using lat/lon masks (WRF grid is Lambert Conformal, so
# we mask on the 2D lat/lon coordinate arrays rather than slicing y/x) ─────
lat_mask    = (wrf_da.lat >= lat0) & (wrf_da.lat <= lat1)
lon_mask    = (wrf_da.lon >= lon0) & (wrf_da.lon <= lon1)
region_mask = lat_mask & lon_mask

reg_wrf_da   = wrf_da.where(region_mask, drop=True)
reg_hist_wrf = reg_wrf_da.sel(time=slice("1981", "2010"))

reg_ssp_wrf = ssp_wrf["prec"].where(region_mask, drop=True).sel(
    sim=[s for s in reg_wrf_da.sim.values if s in ssp_wrf.sim.values]
)

# ── 99th percentile and relative change, spatially averaged over the sub-region ──
reg_hist_p99 = ck.load(reg_hist_wrf.chunk({"time": -1}).quantile(0.99, dim="time").squeeze(), progress_bar=True)
reg_ssp_p99  = ck.load(reg_ssp_wrf.chunk({"time": -1}).quantile(0.99, dim="time").squeeze(), progress_bar=True)

reg_hist_p99_scalar = ck.load(reg_hist_p99.mean(["y", "x"], skipna=True), progress_bar=True)
reg_ssp_p99_scalar  = ck.load(reg_ssp_p99.mean(["y", "x"],  skipna=True), progress_bar=True)
reg_wrf_delta = ck.load(((reg_ssp_p99_scalar - reg_hist_p99_scalar) / reg_hist_p99_scalar * 100), progress_bar=True)

In [ ]:
from climakitae.explore.uncertainty import _area_wgt_average

# Area-average the CMIP6 ensemble spread over the same sub-region.
# CMIP6 regridded data has y=lat degrees, x=lon degrees -- a direct sel()
# slice (not a lat/lon mask) works here since the grid is already regular.
cae_rel_delta = ck.load((delta_cae_p99 / hist_cae_p99 * 100), progress_bar=True)
reg_cae_delta = ck.load(_area_wgt_average(cae_rel_delta.sel(y=slice(lat0, lat1), x=slice(lon0, lon1))), progress_bar=True)

In [ ]:
# Map from clean CMIP6 model name -> normalized WRF `sim` name.
# Only models with a matching single-member downscaled WRF run get a dot;
# models without WRF coverage (e.g. CESM2, CNRM-ESM2-1, FGOALS-g3 here)
# still get a CMIP6 ensemble bar, just no gold dot.
wrf_lookup = {
    "EC-Earth3":     "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    "EC-Earth3-Veg": "wrf_ucla_ec-earth3-veg_ssp370_r1i1p1f1",
    "MIROC6":        "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    "MPI-ESM1-2-HR": "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
    "TaiESM1":       "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
}

sims = np.unique(reg_cae_delta.simulation.values)
bar_tops, bar_floors, medians, wrf_vals = [], [], [], []

for sim in sims:
    mask = reg_cae_delta.simulation == sim
    ens  = reg_cae_delta["pr"].isel(member_id=mask)
    bar_tops.append(float(ens.max(skipna=True)))
    bar_floors.append(float(ens.min(skipna=True)))
    medians.append(float(ens.median(skipna=True)))

    wrf_sim = wrf_lookup.get(sim)
    if wrf_sim is None or wrf_sim not in reg_wrf_delta.sim.values:
        wrf_vals.append(float("nan"))
    else:
        wrf_vals.append(float(reg_wrf_delta.sel(sim=wrf_sim)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.vlines(
    x=sims, ymax=bar_tops, ymin=bar_floors,
    linewidths=22, color="#4472A8", alpha=0.35,
    label="CMIP6 ensemble range\n(internal variability)",
)
ax.scatter(
    sims, medians,
    zorder=3, color="#4472A8", s=600, marker="_", linewidths=2.5,
    label="CMIP6 ensemble median",
)
ax.scatter(
    sims, wrf_vals,
    zorder=4, edgecolors="#1a1a1a", facecolors="#C8A83A",
    s=160, label="Downscaled WRF member",
)
ax.axhline(0, color="#aaaaaa", lw=0.8, ls="dashed")
ax.set_ylabel("% change in 99th-percentile monthly precipitation", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title(
    f"Internal variability vs. inter-model spread \u2014 {warm_level}\u00b0C warming (SSP3-7.0)\n"
    "Sub-region average",
    fontsize=11, pad=10,
)
ax.legend(framealpha=0.9, fontsize=9,
          bbox_to_anchor=(1.02, 0.9), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=25, labelsize=9)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# CMIP6 ensemble range (vertical bars, drawn as thick lines)
for sim, top, floor in zip(sims, bar_tops, bar_floors):
    fig.add_trace(go.Scatter(
        x=[sim, sim],
        y=[floor, top],
        mode="lines",
        line=dict(color="#4472A8", width=22),
        opacity=0.35,
        showlegend=False,
        hoverinfo="skip",
    ))

# Add one dummy trace just for the range legend entry
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines",
    line=dict(color="#4472A8", width=10),
    opacity=0.35,
    name="CMIP6 ensemble range<br>(internal variability)",
))

# Ensemble median (horizontal tick marks)
fig.add_trace(go.Scatter(
    x=sims, y=medians,
    mode="markers",
    marker=dict(symbol="line-ew", size=22, color="#4472A8", line=dict(width=3)),
    name="CMIP6 ensemble median",
    hovertemplate="%{x}<br>Median: %{y:.1f}%<extra></extra>",
))

# Downscaled WRF member
fig.add_trace(go.Scatter(
    x=sims, y=wrf_vals,
    mode="markers",
    marker=dict(
        symbol="circle", size=14,
        color="#C8A83A", line=dict(color="#1a1a1a", width=1.5),
    ),
    name="Downscaled WRF member",
    hovertemplate="%{x}<br>WRF: %{y:.1f}%<extra></extra>",
))

# Zero reference line
fig.add_hline(y=0, line_dash="dash", line_color="#aaaaaa", line_width=1)

fig.update_layout(
    title=dict(
        text=(
            f"Internal variability vs. inter-model spread \u2014 {warm_level}\u00b0C warming (SSP3-7.0)"
            "<br><sup>Sub-region average</sup>"
        ),
        font=dict(size=14),
    ),
    xaxis_title="Model",
    yaxis_title="% change in 99th-percentile monthly precipitation",
    plot_bgcolor="white",
    width=850,
    height=500,
    legend=dict(x=1.02, y=0.9, xanchor="left"),
    xaxis=dict(tickangle=-25),
)

fig.show()
fig.write_html("../uncertainty/figures/html/internal_var_vs_model_spread.html")
fig.write_image("../uncertainty/figures/static/internal_var_vs_model_spread.png", scale=2)

**Key observation:** The blue bars (internal variability within each
model) are typically **larger** than the spread between model medians
(inter-model spread). For extreme precipitation, choosing a different
starting condition within the same model produces more spread than choosing
a completely different model entirely — a striking and counterintuitive
result that underscores why internal variability cannot be ignored for this
variable.

The gold WRF dot sits somewhere within the blue bar for most models — where
exactly depends on which ensemble member happened to be selected for
downscaling. A WRF dot near the top of a model's blue bar does not mean
that model projects more precipitation change than others; it means that
particular member happened to draw from the upper tail of natural
variability. **Conclusions drawn from a single WRF member are therefore
sensitive to which member was chosen, not just to differences in model
physics.**


### Step 2c: Ridgeline — distribution shape per model

The range chart shows only the min–max and median. The ridgeline shows the
**full probability density** of each model's ensemble, making it possible
to see whether a model's distribution is narrow and peaked, wide and flat,
or skewed. The dotted line marks the ensemble median; the gold dot marks
where the single downscaled WRF member lands on that distribution.

Wider, flatter ridges indicate higher internal variability within that
model. Narrower, taller ridges indicate that ensemble members cluster more
tightly — meaning the forced response is more consistently captured
regardless of which member you happen to use.


In [ ]:
MODEL_COLORS = [
    "#4472A8", "#C0664A", "#5A9E6F", "#8B67B5",
    "#C8A83A", "#3D9BAA", "#B84D7A", "#6B7EC2",
]

offset_step = 1.2
n   = len(sims)
fig, ax = plt.subplots(figsize=(9, 0.9 * n + 1.5))

for idx, (sim, color) in enumerate(zip(sims, MODEL_COLORS)):
    mask = reg_cae_delta.simulation == sim
    ens  = reg_cae_delta["pr"].isel(member_id=mask)
    vals = ens.values.flatten()
    vals = vals[~np.isnan(vals)]
    if len(vals) < 2:
        continue

    kde    = gaussian_kde(vals, bw_method=0.5)
    x_grid = np.linspace(vals.min() - 5, vals.max() + 5, 300)
    y_kde  = kde(x_grid)
    y_kde  = y_kde / y_kde.max()
    base   = idx * offset_step

    ax.fill_between(x_grid, base, base + y_kde, color=color, alpha=0.35, zorder=n - idx)
    ax.plot(x_grid, base + y_kde, color=color, lw=1.5, zorder=n - idx)

    med   = float(np.median(vals))
    med_y = kde(med)[0] / y_kde.max()
    ax.plot([med, med], [base, base + med_y],
            color=color, lw=1.2, ls="dashed", alpha=0.85, zorder=n - idx + 1)

    # WRF single member -- select directly by the normalized sim name
    wrf_sim = wrf_lookup.get(sim)
    if wrf_sim is not None and wrf_sim in reg_wrf_delta.sim.values:
        wrf_val = float(reg_wrf_delta.sel(sim=wrf_sim))
        wrf_y   = kde(wrf_val)[0] / y_kde.max()
        ax.scatter([wrf_val], [base + wrf_y], color="#C8A83A",
                   edgecolors="#1a1a1a", s=60, zorder=n, linewidths=0.8)

    ax.text(-0.52, base + 0.08, sim,
            ha="right", va="bottom", fontsize=9,
            color=color, fontweight="medium",
            transform=ax.get_yaxis_transform())

legend_elements = [
    mpatches.Patch(color="#4472A8", alpha=0.4, label="Ensemble distribution"),
    Line2D([0], [0], ls="dashed", color="#555", lw=1.2, label="Ensemble median"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#C8A83A",
           markeredgecolor="#1a1a1a", markersize=7, label="Downscaled WRF single member"),
]
ax.legend(handles=legend_elements, fontsize=9, loc=0, framealpha=0.9)
ax.axvline(0, color="#aaaaaa", lw=0.8, ls="dashed")
ax.set_xlabel("% change in 99th-percentile monthly precipitation", fontsize=11)
ax.set_title(
    f"Within-model distribution of ensemble members \u2014 {warm_level}\u00b0C warming (SSP3-7.0)",
    fontsize=11, pad=10,
)
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(labelsize=10)
ax.set_ylim(-0.3, n * offset_step + 0.5)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

def _hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"

MODEL_COLORS = [
    "#4472A8", "#C0664A", "#5A9E6F", "#8B67B5",
    "#C8A83A", "#3D9BAA", "#B84D7A", "#6B7EC2",
]

offset_step = 1.2
n = len(sims)
fig = go.Figure()

for idx, (sim, color) in enumerate(zip(sims, MODEL_COLORS)):
    mask = reg_cae_delta.simulation == sim
    ens  = reg_cae_delta["pr"].isel(member_id=mask)
    vals = ens.values.flatten()
    vals = vals[~np.isnan(vals)]
    if len(vals) < 2:
        continue

    kde     = gaussian_kde(vals, bw_method=0.5)
    x_grid  = np.linspace(vals.min() - 5, vals.max() + 5, 300)
    density = kde(x_grid)            # raw density, kept for hover text
    y_kde   = density / density.max()  # rescaled to [0, 1] for visual stacking
    base    = idx * offset_step

    # 1) Baseline trace FIRST -- invisible, just establishes the "floor"
    fig.add_trace(go.Scatter(
        x=x_grid, y=[base] * len(x_grid),
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))

    # 2) Ridge curve SECOND -- fill="tonexty" fills down to the trace just added.
    #    Visible line kept thin; hoverinfo turned off here since the wide
    #    invisible hit-area trace below handles hovering instead.
    fig.add_trace(go.Scatter(
        x=x_grid, y=base + y_kde,
        mode="lines",
        line=dict(color=color, width=1.5),
        fill="tonexty",
        fillcolor=_hex_to_rgba(color, 0.35),
        showlegend=False,
        hoverinfo="skip",
    ))

    # 3) Wide invisible hit-area on top of the ridge curve, for easier hovering
    fig.add_trace(go.Scatter(
        x=x_grid, y=base + y_kde,
        mode="lines",
        line=dict(color="rgba(0,0,0,0)", width=15),
        showlegend=False,
        customdata=density,
        hovertemplate=(
            f"<b>{sim}</b><br>"
            "% change: %{x:.1f}<br>"
            "Relative density: %{customdata:.3f}"
            "<extra></extra>"
        ),
    ))

    # Ensemble median (dashed vertical tick)
    med   = float(np.median(vals))
    med_y = kde(med)[0] / density.max()
    fig.add_trace(go.Scatter(
        x=[med, med], y=[base, base + med_y],
        mode="lines",
        line=dict(color=color, dash="dot", width=0.85),
        opacity=0.85,
        showlegend=False,
        hovertemplate=f"{sim} median: %{{x:.1f}}%<extra></extra>",
    ))

    # WRF single member
    wrf_sim = wrf_lookup.get(sim)
    if wrf_sim is not None and wrf_sim in reg_wrf_delta.sim.values:
        wrf_val = float(reg_wrf_delta.sel(sim=wrf_sim))
        wrf_y   = kde(wrf_val)[0] / density.max()
        fig.add_trace(go.Scatter(
            x=[wrf_val], y=[base + wrf_y],
            mode="markers",
            marker=dict(symbol="circle", size=10, color="#C8A83A",
                        line=dict(color="#1a1a1a", width=1.2)),
            showlegend=False,
            hovertemplate=f"{sim} WRF: %{{x:.1f}}%<extra></extra>",
        ))

    # Model label on the left
    fig.add_annotation(
        x=-0.02, y=base + 0.08,
        xref="paper", yref="y",
        text=sim, showarrow=False,
        font=dict(size=11, color=color),
        xanchor="right",
    )

# Zero reference line
fig.add_vline(x=0, line_dash="dash", line_color="#aaaaaa", line_width=1)

# Manual legend entries (since the real traces have showlegend=False)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(size=12, color="#4472A8", opacity=0.4, symbol="square"),
    name="Ensemble distribution"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#555", dash="dot", width=0.85),
    name="Ensemble median"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(symbol="circle", size=10, color="#C8A83A", line=dict(color="#1a1a1a", width=1.2)),
    name="Downscaled WRF single member"))

fig.update_layout(
    title=dict(
        text=f"Within-model distribution of ensemble members \u2014 {warm_level}\u00b0C warming (SSP3-7.0)",
        font=dict(size=14),
    ),
    xaxis_title="% change in 99th-percentile monthly precipitation",
    yaxis=dict(showticklabels=False, range=[-0.3, n * offset_step + 0.5]),
    plot_bgcolor="white",
    width=700,
    height=int(90 * n + 150),
    legend=dict(x=0.85, y=0.95, xanchor="right"),
    margin=dict(l=120),
    hovermode="closest",
)
fig.show()
fig.write_html("../uncertainty/figures/html/warming_ridgeplots.html")
fig.write_image("../uncertainty/figures/static/warming_ridgeplots.png", scale=2)

Wide, flat distributions indicate models with high internal
variability — the single WRF member (gold dot) could have landed anywhere
in that distribution, and a different random seed could easily have
produced a very different downscaled outcome. Narrow, peaked distributions
indicate models where ensemble members cluster tightly, meaning the forced
response is more consistently captured regardless of which member you
happen to use.

Three models in this chart (CESM2, CNRM-ESM2-1, FGOALS-g3) show only the
blue distribution with no gold dot — no corresponding downscaled WRF member
is available for those models, so we can only see their CMIP6 ensemble
spread, not where a single realization happened to land.


## Why internal variability matters for extremes

A 99th-percentile statistic is, by construction, computed from a small
number of extreme months. With only ~360 historical months and a few
hundred future months *per model*, the sample feeding the percentile
estimate is small — and small samples are noisy. If we compute statistics
from a **single ensemble member per model** (as the WRF downscaled product
gives us), our estimate of "how precipitation extremes change" is entangled
with that one particular realization of internal variability, not just the
forced climate-change signal — exactly what Steps 2a–2c demonstrated above.

The practical question this raises: when comparing historical vs. future
precipitation extremes, **is the difference we see real (a forced signal) or
could it just as easily be internal noise?** Answering that requires a
statistical significance test — the focus of Step 3.


## Step 3: Handling high internal variability — data pooling

When internal variability is large relative to the forced signal (as Step 2
demonstrated for extreme precipitation), computing statistics from a single
ensemble member per model gives unreliable results — the estimate depends
heavily on which member/model was chosen, not on the underlying climate
physics.

The recommended approach is to **pool all models together** before computing
statistics, treating each model-month as one independent sample. This
inflates the effective sample size (here, by the number of models — a 7×
factor for the 7 sims we have in common between `hist_wrf` and `ssp_wrf`),
which:
- Gives a more stable estimate of the underlying distribution
- Makes significance tests more conservative and trustworthy — see Step 3e
  for why testing on a *multi-model mean* tends to **overstate** confidence

We compare two approaches side by side:
1. **Multi-model mean (MMM)** — average across models first, then test
2. **Pooled** — keep every model-month as a separate sample, then test

### A critical pitfall: the seasonal cycle

Before running any significance test on raw monthly precipitation, we must
remove the **seasonal cycle** (deseasonalize). California precipitation is
sharply seasonal — wet in winter, near-zero in summer. If we feed raw
monthly values straight into a distribution test, the test will detect the
seasonal cycle itself as "significant," not genuine climate change, because
the historical and future periods may simply sample different numbers of
wet-season vs. dry-season months. This single issue can make significance
maps wildly misleading (we'll see a concrete demonstration in Step 3b) — so
`_deseasonalize_monthly` (defined later in Step 3) is applied before every
KS test in this notebook.

> **Gotcha — deseasonalizing with the `warming_level` processor's output.**
> The future dataset's native time axis is `time_delta` (a relative integer
> index), not a real datetime, so `.dt.month` cannot be used directly on it.
> The cells immediately below rebuild a fresh future dataset (`box_ssp_wrf`)
> using an explicit warming-level lookup table and a manually assigned real
> calendar axis — rather than relying on climakitae's
> `add_dummy_time_to_wl()` helper, which assigns the SAME fixed dummy
> calendar range (starting `2000-01-01`) to every simulation regardless of
> its true start month. Using `.dt.month` on that dummy time would silently
> misalign which true season gets grouped together across models, and
> produced wildly inflated, spurious significance (tens of thousands of
> "significant" points) when this analysis was first built without this
> fix.

**The next six cells implement this fix.** They retrieve a fresh full-domain
WRF dataset, look up the year each simulation crosses +2.5°C warming
(relative to a 1981–2010 baseline), slice out a 30-year window centred on
that year for each simulation individually, and assign each one a clean,
real calendar time axis (2070–2100) so that `.dt.month` — and therefore
deseasonalizing — works correctly and consistently across every model.


In [ ]:
from climakitae.core.data_interface import DataParameters
from climakitae.explore.uncertainty import get_ensemble_data, CmipOpt, get_warm_level

selections = DataParameters()
selections.downscaling_method = 'Dynamical'
selections.scenario_historical = ['Historical Climate']
selections.scenario_ssp = ['SSP 3-7.0']
selections.append_historical = True
selections.variable = 'Precipitation (total)'
selections.time_slice = (1980, 2100)
selections.resolution = '9 km'
selections.area_subset='states'
selections.cached_area = ['CA']
selections.area_average = 'No'
selections.timescale = 'monthly'

cmip_names = [
    'EC-Earth3', 'EC-Earth3-Veg', 'CESM2', 'CNRM-ESM2-1',  'FGOALS-g3', 
    'MIROC6', 'TaiESM1', 'MPI-ESM1-2-HR'
]

box_wrf_ds = selections.retrieve().squeeze()

In [ ]:
gwl_file_all = GWL_1981_2010_FILE
gwl_times_all = read_csv_file(gwl_file_all, index_col=[0, 1, 2])
# TODO Add information on a more complete list of ensemble members of
# EC-Earth3 to cover internal variability notebook needs
gwl_file_ece3 = "data/gwl_1981-2010ref_EC-Earth3_ssp370.csv"
gwl_times_ece3 = read_csv_file(gwl_file_ece3, index_col=[0, 1, 2])
gwl_times = pd.concat([gwl_times_all, gwl_times_ece3]).drop_duplicates()
gwl_times = gwl_times.reset_index()
gwl_times["GCM_run"] = "WRF_" + gwl_times["GCM"] + "_" + gwl_times["run"]
gwl_times = gwl_times.query("scenario=='ssp370'").set_index("GCM_run").drop(columns=["GCM","run","scenario"])['2.5']
gwl_times = pd.to_datetime(gwl_times)

In [ ]:
gwl_times = gwl_times.loc[gwl_times.index.isin(box_wrf_ds.simulation.values)]
start_time = pd.to_datetime([f"{y-15}-01-01 00:00:00" for y in gwl_times.dt.year])
end_time = pd.to_datetime([f"{y+15}-01-01 00:00:00" for y in gwl_times.dt.year])
gwl_times = pd.DataFrame({
    "central_time":gwl_times.values, 
    "start_time":start_time.values, 
    "end_time":end_time.values
}, index=gwl_times.index)

In [ ]:
time = pd.date_range("2070-01-01 00:00:00", "2100-01-01 00:00:00", freq="MS")

In [ ]:
box_ssp_wrf = []
for sim in box_wrf_ds.simulation.values:
    try:
        t0 = str(gwl_times.loc[sim,'start_time'][0])
        t1 = str(gwl_times.loc[sim,'end_time'][0])
    except:
        t0 = str(gwl_times.loc[sim,'start_time'])
        t1 = str(gwl_times.loc[sim,'end_time'])
    tmp = box_wrf_ds.sel(simulation=sim, time=slice(t0, t1))
    is_feb29 = (tmp.time.dt.month == 2) & (tmp.time.dt.day == 29)
    tmp = tmp.where(~is_feb29, drop=True)
    if len(tmp.time)<350:
        continue
    else:
        tmp = tmp.assign_coords(time=time)
        box_ssp_wrf.append(tmp)
box_ssp_wrf = xr.concat(box_ssp_wrf, dim="simulation")

In [ ]:
box_hist_wrf = box_wrf_ds.sel(time=slice('1981','2010'))
box_ssp_wrf = box_ssp_wrf.sel(time=slice('2070','2099'))

common_sims = np.intersect1d(box_hist_wrf.simulation.values, box_ssp_wrf.simulation.values)
box_hist_wrf = box_hist_wrf.sel(simulation=common_sims)
box_ssp_wrf = box_ssp_wrf.sel(simulation=common_sims)

In [ ]:
def _deseasonalize_monthly(ds, month_coord="month", dim="time"):
    """Remove the monthly climatology, returning anomalies.

    Subtracts the mean for each calendar month (1-12) at every grid cell,
    so the seasonal cycle no longer dominates downstream statistical tests
    (e.g. the KS test below). Groups by a pre-computed integer `month`
    coordinate (1-12) rather than `.dt.month`, since data coming from the
    `warming_level` processor uses a `time_delta` dimension with no real
    datetime to call `.dt.month` on -- see Gotcha #3 above. For data that
    DOES have a real datetime `time` coordinate (e.g. the historical
    baseline retrieved via a plain time slice), the `month` coordinate
    attached in Step 1 (`time_delta_to_month`) is equally valid here, since
    it agrees with `.dt.month` by construction.

    Parameters
    ----------
    ds : xr.DataArray
        Input data with an integer `month_coord` coordinate (1-12) along `dim`
    month_coord : str
        Name of the integer month coordinate, default "month"
    dim : str
        Name of the dimension being grouped over (e.g. "time" or "time_delta")

    Returns
    -------
    xr.DataArray
        Deseasonalized anomalies, same shape as input
    """
    if month_coord not in ds.coords:
        raise ValueError(
            f"\'{month_coord}\' coordinate not found. Available coords: {list(ds.coords)}"
        )
    climatology = ds.groupby(month_coord).mean(dim)
    anomalies = ds.groupby(month_coord) - climatology
    
    return anomalies.drop_vars(month_coord, errors="ignore")

In [ ]:
def get_ks_pval_df(
    sample1: xr.Dataset | xr.DataArray,
    sample2: xr.Dataset | xr.DataArray,
    sig_lvl: float = 0.05,
) -> pd.DataFrame:
    """Two-sample Kolmogorov-Smirnov test at every spatial grid point.

    Compares the full DISTRIBUTION of sample1 vs. sample2 at each (y, x)
    cell -- not just their means -- and returns only the cells where the
    difference is statistically significant (p < sig_lvl).

    Two important implementation details:

    1. `sample1`/`sample2` must NOT be aligned by their time/index
       coordinate VALUES. The KS test compares two independent
       distributions, not paired/matched samples -- if sample1 covers
       historical dates (e.g. 1981-2010) and sample2 covers a future
       warming-level window (e.g. 2047-2093), aligning on coordinate
       values (xarray\'s default) finds zero date overlap and silently
       empties the core dimension, making every p-value NaN. We avoid
       this by dropping the core_dim coordinate before aligning, and
       aligning only on the spatial dimension.
    2. NaNs must be filtered out inside the per-cell test function itself.
       `xr.apply_ufunc` does not do this automatically, and even one NaN
       in a sample (e.g. from a model with incomplete coverage at a given
       grid cell) can silently corrupt `scipy.stats.kstest`\'s result for
       that cell.

    Parameters
    ----------
    sample1 : xr.Dataset or xr.DataArray
        First sample for comparison (e.g. historical anomalies)
    sample2 : xr.Dataset or xr.DataArray
        Second sample for comparison (e.g. future anomalies)
    sig_lvl : float
        Alpha level for statistical significance, default 0.05

    Returns
    -------
    pd.DataFrame
        Columns are lon, lat, and p_value; only retains spatial points
        where p_value < sig_lvl
    """
    if isinstance(sample1, xr.Dataset):
        sample1 = sample1[list(sample1.data_vars)[0]]
    if isinstance(sample2, xr.Dataset):
        sample2 = sample2[list(sample2.data_vars)[0]]

    sample1 = sample1.stack(allpoints=["y", "x"]).squeeze()
    sample2 = sample2.stack(allpoints=["y", "x"]).squeeze()

    core_dim_candidates = [d for d in sample1.dims if d != "allpoints"]
    if len(core_dim_candidates) != 1:
        raise ValueError(
            f"Expected exactly one non-spatial dimension after stacking, "
            f"got {core_dim_candidates}. Ensure inputs have shape (time/index, y, x)."
        )
    core_dim = core_dim_candidates[0]

    if core_dim in sample1.coords:
        sample1 = sample1.drop_vars(core_dim)
    if core_dim in sample2.coords:
        sample2 = sample2.drop_vars(core_dim)
    sample1, sample2 = xr.align(sample1, sample2, join="inner", exclude=[core_dim])

    def ks_stat_2sample(s1, s2):
        """Run a two-sample KS test at one grid cell, filtering NaNs first."""
        s1 = s1[~np.isnan(s1)]
        s2 = s2[~np.isnan(s2)]
        try:
            d_statistic, p_value = stats.kstest(s1, s2)
        except (ValueError, ZeroDivisionError):
            d_statistic, p_value = np.nan, np.nan
        return d_statistic, p_value

    _, p_value = xr.apply_ufunc(
        ks_stat_2sample,
        sample1,
        sample2,
        input_core_dims=[[core_dim], [core_dim]],
        exclude_dims=set((core_dim,)),
        output_core_dims=[[], []],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float, float],
    )

    p_value = p_value.compute()
    p_value = p_value.rename("p_value")
    p_df = p_value.unstack("allpoints")
    p_df = p_df.to_dataframe().reset_index()
    p_df = p_df.dropna(subset=["p_value"])
    p_df = p_df[["lat", "lon", "p_value"]]
    p_df = p_df.loc[:, ["lon", "lat", "p_value"]]

    return p_df[p_df["p_value"] < sig_lvl]

### Step 3a: Build the multi-model-mean and pooled comparisons

Both approaches start from the same `box_hist_wrf` / `box_ssp_wrf` data (the
full-California WRF data built above, with matching `simulation` values and
a real calendar time axis on each), but diverge in how they treat the
`simulation` dimension before computing the 99th-percentile statistic:

- **MMM**: average across `simulation` first (1 series per grid cell), then
  compute the percentile
- **Pooled**: keep every (`simulation`, `time`) combination as a separate
  sample, stack them into one long `index` dimension, then compute the
  percentile across the full pooled sample

The print statement below confirms how many simulations and months feed
each approach before we proceed — useful as a sanity check if the later
significance counts look unexpectedly large or small.


In [ ]:
n_sims = len(box_hist_wrf.simulation.values)
print(f"Common simulations for Step 3: {n_sims}")
print(f"hist_wrf dims: {box_hist_wrf.dims}, sizes: {box_hist_wrf.sizes}")
print(f"ssp_wrf  dims: {box_ssp_wrf.dims}, sizes: {box_ssp_wrf.sizes}")

In [ ]:
# ── Multi-model mean approach ─────────────────────────────────────────────
box_hist_p99_mmm = ck.load(
    box_hist_wrf.chunk({"time": -1}).quantile(0.99, dim="time").squeeze().unify_chunks(),
    progress_bar=True
)
box_ssp_p99_mmm = ck.load(
    box_ssp_wrf.chunk({"time": -1}).quantile(0.99, dim="time").squeeze().unify_chunks(),
    progress_bar=True
)

hist_mmm  = box_hist_p99_mmm.mean(dim="simulation").compute().squeeze()
ssp_mmm   = box_ssp_p99_mmm.mean(dim="simulation").compute().squeeze()
delta_mmm = (ssp_mmm - hist_mmm).compute().squeeze()

In [ ]:
# ── Pooled approach -- stack all simulations and time into one dimension ──
hist_pool = box_hist_wrf.stack(index=["simulation", "time"]).compute()
ssp_pool  = box_ssp_wrf.stack(index=["simulation", "time"]).compute()

hist_pool_p99 = ck.load(hist_pool.chunk({"index": -1}).quantile(0.99, dim="index").squeeze(), progress_bar=True)
ssp_pool_p99  = ck.load(ssp_pool.chunk({"index": -1}).quantile(0.99, dim="index").squeeze(), progress_bar=True)
delta_pool    = (ssp_pool_p99 - hist_pool_p99).compute()

### Step 3b: KS significance test — deseasonalized

We deseasonalize using the real calendar `time` axis built in Step 3
(`box_hist_wrf` and `box_ssp_wrf` both have genuine datetimes, so
`.dt.month` is reliable here — this is exactly the case the Gotcha above
warns about, and we've already worked around it by constructing
`box_ssp_wrf` with a real calendar axis rather than `time_delta`). For the
pooled case, each simulation's series is deseasonalized individually (the
seasonal cycle is removed per-model, per grid cell) before stacking into
the pooled sample.

The next four cells compute the 99th-percentile change two ways — first via
the multi-model mean (`delta_percentile_mmm`), then via the pooled sample
(`delta_wrf_pool_perc`) — and run the KS significance test (`get_ks_pval_df`,
defined in Step 3 above) on both. `mmm_p_df` and `pooled_p_df` each contain
only the grid cells where the test detects a statistically significant
difference (p < 0.05) between historical and future precipitation
distributions.


In [ ]:
box_cads_hist_percentile = box_hist_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

box_cads_ssp_percentile = box_ssp_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

In [ ]:
box_cads_delta_percentile = (((box_cads_ssp_percentile
                              - box_cads_hist_percentile)
                             /box_cads_hist_percentile
                             )*100).compute()

In [ ]:
hist_wrf_pool = box_hist_wrf.stack(index=['simulation','time'])
hist_wrf_pool = hist_wrf_pool.compute()

ssp_percentile_mmm = box_cads_ssp_percentile.mean(
    dim="simulation").compute().squeeze()
hist_percentile_mmm = box_cads_hist_percentile.mean(
    dim="simulation").compute().squeeze()
delta_percentile_mmm = (ssp_percentile_mmm 
            - hist_percentile_mmm).compute().squeeze()
hist_wrf_pool = box_hist_wrf.stack(index=['simulation','time'])
ssp_wrf_pool = box_ssp_wrf.stack(index=['simulation','time'])

In [ ]:
hist_wrf_pool_perc = hist_wrf_pool.chunk(
    dict(index=-1)).quantile([.99],
    dim='index').compute().squeeze()

ssp_wrf_pool_perc = ssp_wrf_pool.chunk(
    dict(index=-1)).quantile([.99],
    dim='index').compute().squeeze()

delta_wrf_pool_perc = (ssp_wrf_pool_perc
                       - hist_wrf_pool_perc
                      ).compute()

hist_wrf_pool = hist_wrf_pool.compute()
ssp_wrf_pool = ssp_wrf_pool.compute()
pooled_p_df = get_ks_pval_df(hist_wrf_pool, ssp_wrf_pool)

hist_wrf_mmm = box_hist_wrf.mean(dim='simulation').rename({'time':'index'}).compute()
ssp_wrf_mmm = box_ssp_wrf.mean(dim='simulation').rename({'time':'index'}).compute()
mmm_p_df = get_ks_pval_df(hist_wrf_mmm, ssp_wrf_mmm)

**What to expect:** with deseasonalization and warming-level windowing
applied correctly, the **pooled** significance map should show fewer, more
spatially clustered significant points than the **multi-model mean** map —
concentrated where the magnitude of change is largest (e.g. the strongest
wetting/drying signal). The mean approach tends to over-detect significance
almost everywhere, because averaging across models suppresses noise so
aggressively that even small, not-very-robust shifts register as
"significant." This is why **the pooled result, not the mean result, should
be treated as the primary, most defensible significance estimate** — a key
takeaway of this notebook.

If you instead see the **pooled** result showing dramatically *more*
significant points than the mean (this jumped to tens of thousands of
points while debugging this exact notebook), it's a strong sign that either
the warming-level window wasn't applied correctly (check the sample-size
print statement above — it should be a modest multiple of ~360, not several
thousand) or the seasonal cycle wasn't removed correctly for every
simulation (see the Gotcha in Step 3's introduction).


### Step 3c: Comparison maps — cartopy (static / publication-style)

Cartopy gives clean, publication-ready maps with real geographic
projections, coastlines, and state borders. This is the version to use for
reports, papers, or any static figure.


In [ ]:
def make_precip_map_cartopy(data, title, vmin, vmax, diverging=False,
                             ax=None, sig_df=None,
                             extent=[-125, -113, 32, 43]):
    """Precipitation map with cartopy projection, coastlines, optional KS stippling.

    Parameters
    ----------
    data : xr.Dataset or xr.DataArray
        WRF data with 2D lat/lon coordinates (Lambert Conformal grid)
    title : str
        Map title
    vmin, vmax : float or None
        Colorbar limits; if both None, computed automatically from the data
    diverging : bool
        Use a diverging (blue-white-red) colormap if True, sequential otherwise
    ax : cartopy GeoAxes, optional
        Axes to plot on; a new standalone figure is created if None
    sig_df : pd.DataFrame, optional
        KS significance points with \'lon\' and \'lat\' columns -- plotted as
        small stippled dots marking statistically significant grid cells
    extent : list
        [lon_min, lon_max, lat_min, lat_max] in degrees, default covers California

    Returns
    -------
    cartopy GeoAxes
    """
    if isinstance(data, xr.Dataset):
        data = data[list(data.data_vars)[0]]

    cmap_name = "ae_diverging_r" if diverging else "ae_blue"
    cmap_hex  = read_ae_colormap(cmap=cmap_name, cmap_hex=True)
    cmap      = LinearSegmentedColormap.from_list(cmap_name, cmap_hex)

    vals = np.array(data.values, dtype=float)
    if vmin is None and vmax is None:
        if diverging:
            absmax = float(np.nanpercentile(np.abs(vals), 98))
            vmin, vmax = -absmax, absmax
        else:
            vmin = float(np.nanpercentile(vals, 2))
            vmax = float(np.nanpercentile(vals, 98))

    display_crs = ccrs.PlateCarree()
    standalone  = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 6), subplot_kw={"projection": display_crs})

    # im = ax.pcolormesh(
    im = ax.contourf(
        data["lon"].values, data["lat"].values, vals,
        cmap=cmap, vmin=vmin, vmax=vmax,
        transform=display_crs,
        # shading="auto",
    )

    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="k")
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, edgecolor="k")
    ax.add_feature(cfeature.STATES,    linewidth=0.3, edgecolor="#555555")
    ax.add_feature(cfeature.OCEAN,     facecolor="#e8f4f8", zorder=0)
    ax.set_extent(extent, crs=display_crs)

    if sig_df is not None and len(sig_df) > 0:
        ax.scatter(
            sig_df["lon"], sig_df["lat"],
            s=0.5, color="k", alpha=0.35,
            transform=display_crs, zorder=5,
        )

    ax.set_title(title, fontsize=10, pad=6)
    plt.colorbar(im, ax=ax, orientation="vertical",
                 label="mm / month", shrink=0.85, pad=0.02)

    if standalone:
        plt.tight_layout()
        plt.show()

    return ax


In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(9, 8),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# Row 1: raw difference maps, no stippling
make_precip_map_cartopy(
    delta_mmm, "Multi-model mean", vmin=-300, vmax=300,
    diverging=True, ax=axes[0, 0]
)
make_precip_map_cartopy(
    delta_pool, "Multi-model pool", vmin=-300, vmax=300,
    diverging=True, ax=axes[0, 1]
)

# Row 2: same maps with KS significance stippling overlaid
make_precip_map_cartopy(
    delta_mmm, "Multi-model mean \u2014 KS significant", vmin=-300, vmax=300,
    diverging=True, ax=axes[1, 0], sig_df=mmm_p_df
)
make_precip_map_cartopy(
    delta_pool, "Multi-model pool \u2014 KS significant", vmin=-300, vmax=300,
    diverging=True, ax=axes[1, 1], sig_df=pooled_p_df
)

fig.suptitle(
    f"Change in 99th-percentile monthly precipitation at {warm_level}\u00b0C warming (SSP3-7.0)",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import cartopy.feature as cfeature

# ── Shared setup ─────────────────────────────────────────────────────────────
vmin, vmax = -250, 250
cmap_hex = read_ae_colormap(cmap="ae_diverging_r", cmap_hex=True)
colorscale = [[i / (len(cmap_hex) - 1), c] for i, c in enumerate(cmap_hex)]

coastline_feature = cfeature.NaturalEarthFeature("physical", "coastline", "10m")
states_feature = cfeature.NaturalEarthFeature("cultural", "admin_1_states_provinces_lines", "10m")

def _geom_to_xy(geometries, extent):
    xs, ys = [], []
    lon_min, lon_max, lat_min, lat_max = extent
    for geom in geometries:
        for line in getattr(geom, "geoms", [geom]):
            coords = list(line.coords)
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
            if max(lons) < lon_min or min(lons) > lon_max or max(lats) < lat_min or min(lats) > lat_max:
                continue
            xs.extend(lons + [None])
            ys.extend(lats + [None])
    return xs, ys

def _get_lonlat_flat(data):
    if isinstance(data, xr.Dataset):
        data = data[list(data.data_vars)[0]]
    lons = np.asarray(data["lon"].values).flatten()
    lats = np.asarray(data["lat"].values).flatten()
    vals = np.asarray(data.values, dtype=float).flatten()

    # Drop NaN points entirely -- Scattergl still tries to render them otherwise
    valid = ~np.isnan(vals)
    return lons[valid], lats[valid], vals[valid]


def _get_xy_grid(data):
    """For go.Contour: needs regular 1D x/y axes. Use the projected x/y
    (not lon/lat, which are 2D/irregular on this Lambert Conformal grid)."""
    if isinstance(data, xr.Dataset):
        data = data[list(data.data_vars)[0]]
    x = np.asarray(data["x"].values)
    y = np.asarray(data["y"].values)
    z = np.asarray(data.values, dtype=float)
    return x, y, z
    

def build_comparison_figure(data_left, title_left, data_right, title_right,
                             sig_df_left=None, sig_df_right=None):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[title_left, title_right],
        horizontal_spacing=0.06,
    )

    for col, (data, title, sig_df) in enumerate(
        [(data_left, title_left, sig_df_left), (data_right, title_right, sig_df_right)], start=1
    ):
        lons_flat, lats_flat, vals_flat = _get_lonlat_flat(data)

        # 1) Data markers
        fig.add_trace(
            go.Scattergl(
                x=lons_flat, y=lats_flat,
                mode="markers",
                marker=dict(
                    size=7, symbol="square",
                    color=vals_flat,
                    colorscale=colorscale,
                    cmin=vmin, cmax=vmax,
                    showscale=(col == 1),
                    colorbar=dict(
                        title=dict(text="mm / month", side="top", font=dict(size=10)),
                        orientation="h",
                        len=0.75, thickness=14,
                        x=0.5, xanchor="center",
                        y=-0.12, yanchor="top",
                        tickfont=dict(size=9),
                    ) if col == 1 else None,
                ),
                showlegend=False,
                hovertemplate="Lon: %{x:.2f}<br>Lat: %{y:.2f}<br>Precip: %{marker.color:.1f} mm/mo<extra></extra>",
            ),
            row=1, col=col,
        )

        # 2) Coastline / state borders
        fig.add_trace(
            go.Scatter(x=coast_x, y=coast_y, mode="lines",
                       line=dict(color="black", width=0.7),
                       showlegend=False, hoverinfo="skip"),
            row=1, col=col,
        )
        fig.add_trace(
            go.Scatter(x=state_x, y=state_y, mode="lines",
                       line=dict(color="#777777", width=0.5),
                       showlegend=False, hoverinfo="skip"),
            row=1, col=col,
        )

        # 3) Significance dots LAST -- drawn on top, made larger/darker for visibility
        if sig_df is not None and len(sig_df) > 0:
            fig.add_trace(
                go.Scattergl(
                    x=sig_df["lon"], y=sig_df["lat"],
                    mode="markers",
                    marker=dict(size=1.5, color="black", opacity=0.50,
                                line=dict(width=0)),
                    showlegend=False,
                    hovertemplate="Significant<br>Lon: %{x:.2f}<br>Lat: %{y:.2f}<extra></extra>",
                ),
                row=1, col=col,
            )

        fig.update_xaxes(showticklabels=False, range=[extent[0], extent[1]], row=1, col=col)
        fig.update_yaxes(showticklabels=False, range=[extent[2], extent[3]], row=1, col=col)

    fig.update_layout(
        width=750, height=480,
        plot_bgcolor="white",
        margin=dict(t=50, l=10, r=10, b=70),
        showlegend=False,  # belt-and-suspenders: suppress legend at the figure level too
    )
    return fig

In [ ]:
fig_no_sig = build_comparison_figure(
    delta_mmm, "Multi-model mean",
    delta_pool, "Multi-model pool",
)

fig_sig = build_comparison_figure(
    delta_mmm, "Multi-model mean \u2014 KS significant",
    delta_pool, "Multi-model pool \u2014 KS significant",
    sig_df_left=mmm_p_df, sig_df_right=pooled_p_df,
)

# ── Tabs via panel ────────────────────────────────────────────────────────────
title_text = (
    f"<h3 style='text-align:center; font-weight:500; color:#1a1a2e;'>"
    f"Change in 99th-percentile monthly precipitation at {warm_level}°C warming (SSP3-7.0)"
    f"</h3>"
)
comparison = pn.Column(
    pn.pane.HTML(title_text),
    pn.Tabs(
        ("No significance", fig_no_sig),
        ("KS significant", fig_sig),
        dynamic=False,
    ),
)
comparison

In [ ]:
comparison.save("../uncertainty/figures/html/ks_99_precip.html", embed=True)
# Static PNGs (one per tab, plotly's own exporter)
fig_no_sig.write_image("../uncertainty/figures/static/ks_99_precip_no_sig.png", scale=2)
fig_sig.write_image("../uncertainty/figures/static/ks_99_precip_sig.png", scale=2)

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Multi-model mean", "Multi-model pool"],
    horizontal_spacing=0.06,
)

no_sig_trace_idx = []
sig_trace_idx = []

def add_panel(data, col, sig_df, group_list, show_colorbar, coloraxis_name):
    lons_flat, lats_flat, vals_flat = _get_lonlat_flat(data)

    fig.add_trace(
        go.Scattergl(
            x=lons_flat, y=lats_flat,
            mode="markers",
            marker=dict(
                size=7, symbol="square",
                color=vals_flat,
                coloraxis=coloraxis_name,   # explicit, separate coloraxis per tab-group
            ),
            showlegend=False,
            visible=True,
            hovertemplate="Lon: %{x:.2f}<br>Lat: %{y:.2f}<br>Precip: %{marker.color:.1f} mm/mo<extra></extra>",
        ),
        row=1, col=col,
    )
    group_list.append(len(fig.data) - 1)

    fig.add_trace(
        go.Scatter(x=coast_x, y=coast_y, mode="lines",
                   line=dict(color="black", width=0.7),
                   showlegend=False, hoverinfo="skip", visible=True),
        row=1, col=col,
    )
    group_list.append(len(fig.data) - 1)

    fig.add_trace(
        go.Scatter(x=state_x, y=state_y, mode="lines",
                   line=dict(color="#777777", width=0.5),
                   showlegend=False, hoverinfo="skip", visible=True),
        row=1, col=col,
    )
    group_list.append(len(fig.data) - 1)

    if sig_df is not None and len(sig_df) > 0:
        fig.add_trace(
            go.Scattergl(
                x=sig_df["lon"], y=sig_df["lat"],
                mode="markers",
                marker=dict(size=1.5, color="black", opacity=0.5, line=dict(width=0)),
                showlegend=False, hoverinfo="skip", visible=True,
            ),
            row=1, col=col,
        )
        group_list.append(len(fig.data) - 1)

    fig.update_xaxes(showticklabels=False, range=[extent[0], extent[1]], row=1, col=col)
    fig.update_yaxes(showticklabels=False, range=[extent[2], extent[3]], row=1, col=col)


# Both groups use the SAME coloraxis ("coloraxis"), since they share the same scale/range
add_panel(delta_mmm,  col=1, sig_df=None,        group_list=no_sig_trace_idx, show_colorbar=True,  coloraxis_name="coloraxis")
add_panel(delta_pool, col=2, sig_df=None,        group_list=no_sig_trace_idx, show_colorbar=False, coloraxis_name="coloraxis")
add_panel(delta_mmm,  col=1, sig_df=mmm_p_df,    group_list=sig_trace_idx,    show_colorbar=True,  coloraxis_name="coloraxis")
add_panel(delta_pool, col=2, sig_df=pooled_p_df, group_list=sig_trace_idx,    show_colorbar=False, coloraxis_name="coloraxis")

n_traces = len(fig.data)

def visibility_mask(active_idx):
    return [i in active_idx for i in range(n_traces)]

fig.update_layout(
    width=750, height=560,
    plot_bgcolor="white",
    margin=dict(t=140, l=10, r=10, b=70),
    showlegend=False,
    title=dict(
        text=f"Change in 99th-percentile monthly precipitation at {warm_level}\u00b0C warming (SSP3-7.0)",
        font=dict(size=15), x=0.5, xanchor="center", y=0.92,
    ),
    # ── Single coloraxis, defined ONCE at the figure level, always present ──
    coloraxis=dict(
        colorscale=colorscale,
        cmin=vmin, cmax=vmax,
        colorbar=dict(
            title=dict(text="mm / month", side="top", font=dict(size=10)),
            orientation="h", len=0.75, thickness=14,
            x=0.5, xanchor="center", y=-0.12, yanchor="top",
            tickfont=dict(size=9),
        ),
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.0, y=1.20,
            xanchor="left", yanchor="top",
            showactive=True,
            buttons=[
                dict(label="No significance",
                     method="update",
                     args=[{"visible": visibility_mask(no_sig_trace_idx)}]),
                dict(label="KS significant",
                     method="update",
                     args=[{"visible": visibility_mask(sig_trace_idx)}]),
            ],
        )
    ],
)

for i in range(n_traces):
    fig.data[i].visible = i in no_sig_trace_idx

fig.show()

In [ ]:
fig.write_html("../uncertainty/figures/html/ks_99_precip.html")

# For static export (per-state snapshot, since a static image can only show one button state at a time):
for i in range(n_traces):
    fig.data[i].visible = i in no_sig_trace_idx
fig.write_image("../uncertainty/figures/static/ks_99_precip_no_sig.png", scale=2)

for i in range(n_traces):
    fig.data[i].visible = i in sig_trace_idx
fig.write_image("../uncertainty/figures/static/ks_99_precip_sig.png", scale=2)

### Reading the maps
This figure shows the pattern the markdown text predicted, with the pooled map now producing a sensible (rather than inflated) significance result. Here's how to read it:

- **The colors (both panels, identical):** Show the change in 99th-percentile monthly precipitation between the warming-level future window and the 1981–2010 historical baseline. Dark blue = substantially wetter extremes; orange/red = drier extremes. Both panels show essentially the same physical pattern — a strong wetting signal running through the Sierra Nevada and northern coastal mountains, with weaker or mixed signal in the south.

- **The stippled dots:** Mark grid cells where the KS test finds the future and historical precipitation distributions are statistically distinguishable (p < 0.05), after deseasonalizing.

### Reading the maps

- The **multi-model mean** (left) shows significance only in a fairly narrow band, concentrated tightly around the strongest part of the Sierra Nevada wetting signal. Most of the state, including areas with visible color change, is *not* stippled.

- The **multi-model pool** (right) shows significance across a much broader area — not just the Sierra Nevada core but extending into the Central Valley, parts of the south, and even some areas where the color signal looks weak or mixed (the southeastern desert region shows dense stippling despite faint coloring).

This is the opposite pattern from what the notebook's "What to expect" callout describes as the *broken* case (pooled showing tens of thousands of spurious significant points everywhere) — but it's also not quite the *idealized* case described either, where pooled should show "fewer, more spatially clustered" significant points than the mean. Here pooled clearly shows **more** significant area than the mean, which the notebook frames as the correct, expected outcome of pooling: the larger effective sample size gives the KS test more statistical power to detect real differences, including subtler ones in the south that the mean approach — with its sample size capped by what's used for the percentile-then-average asymmetric, but the mean approach's deflated variance from averaging — fails to flag.

**The practical takeaway:** Trust the pooled (right) result as the primary map. The multi-model mean understates how much of California shows a detectable shift in extreme precipitation, because averaging across models before testing artificially shrinks the sample's apparent variability and makes the test more conservative than it should be. The pooled approach gives a more complete and statistically honest picture of where the climate signal is actually distinguishable from natural variability.

### Step 3d: Sample size effect — why pooling matters quantitatively

The cell below prints a concrete sample size comparison between the two
approaches, making the statistical argument explicit: pooling doesn't change
the underlying data, but it does change how much of it the significance test
gets to see at once. This is the quantitative explanation for the
significance-pattern difference shown in Step 3c.


In [ ]:
n_sims   = len(box_hist_wrf.simulation.values)
n_months = len(box_hist_wrf.time.values)
n_pool   = n_sims * n_months

print("=" * 52)
print("  SAMPLE SIZE: MULTI-MODEL MEAN vs. POOLED")
print("=" * 52)
print(f"  Models available:           {n_sims}")
print(f"  Months per model:           {n_months}")
print()
print(f"  MMM sample (per grid cell): {n_months} months x {n_sims} model means")
print(f"  Pooled sample:              {n_pool} months ({n_sims} x {n_months})")
print(f"  Pooling factor:             {n_sims}x")
print("=" * 52)
print()
print("  For a 99th-percentile estimate, a minimum of ~200")
print("  samples is typically needed for statistical stability.")

n_months_label = "sufficient" if n_months >= 200 else "marginal"
n_pool_label   = "sufficient" if n_pool >= 200 else "marginal"
print(f"  Single-model sample: {n_months} -> {n_months_label}")
print(f"  Pooled sample:       {n_pool} -> {n_pool_label}")


## Step 4: When does internal variability matter?

Internal variability is not equally important for all variables, regions, or
time horizons. The guidance below summarizes the key rules of thumb that
follow from Steps 1–3.

| Context | Internal variability dominates? | Recommended approach |
|---|---|---|
| Near-term (< 20 yr), any variable | **Yes** | Pool ensemble members; acknowledge large uncertainty |
| Long-term (> 40 yr), temperature | **No** — model/scenario dominate | Multi-model ensemble, keep scenarios separate |
| Long-term (> 40 yr), extreme precipitation | **Yes** | Pool; use KS test with conservative thresholds |
| Regional scale | **Yes** — amplified regionally | Pool; avoid single-model conclusions |
| Global scale | **Partial** | Standard multi-model mean is more defensible |

**Practical checklist for any new internal-variability analysis:**
1. Are you analyzing an **extreme** statistic (percentile, max, count of
   threshold exceedances)? If so, internal variability likely matters —
   don't skip the pooling step.
2. Does your variable have a strong **seasonal cycle** (precipitation,
   especially)? If so, deseasonalize before any significance test — and if
   your data comes from the `warming_level` processor, build a real
   calendar time axis for the future period (as done in Step 3) rather than
   relying on a dummy-time reconstruction.
3. Are you testing significance on a **multi-model mean**? Be skeptical of
   results that look significant almost everywhere — cross-check against
   the pooled result, as in Step 3c.
4. Do you have only **one ensemble member per model**? You're capturing
   inter-model spread but not true within-model internal variability — for
   that, you need a single-model large ensemble, as used in Steps 2a–2c
   above.
5. After pooling, does your sample size match what you'd expect from
   `n_models × window_length`? A mismatch (e.g. one side several times
   larger than expected) usually means the warming-level window wasn't
   applied correctly on one side — check the print statement in Step 3a as
   a sanity check.

> For a quantitative decomposition of how internal variability compares to
> model and scenario uncertainty over time, see `uncertainty_decomposition.ipynb`.

**References:**
- [Hawkins & Sutton (2009) — *BAMS*](https://journals.ametsoc.org/view/journals/bams/90/8/2009bams2607_1.xml):
  the canonical source for the variance decomposition framework used in
  `uncertainty_decomposition.ipynb`
- [IPCC AR5 uncertainty guidance](https://www.ipcc.ch/site/assets/uploads/2017/08/AR5_Uncertainty_Guidance_Note.pdf):
  formal IPCC framework for communicating uncertainty
- [climatewest.ca: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/):
  accessible overview of all three uncertainty sources
- [climatedata.ca: Uncertainty in projections](https://climatedata.ca/resource/uncertainty-in-climate-projections/):
  regional application context
